<a href="https://colab.research.google.com/github/sting909/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sting909/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Contract Question 1 & 3: One Row Definition, Table & Time Window

One Row Definition: One row represents a single unique (url, query, date) combination.

Table Used: hf://datasets/FlyRank/internship-warehouse/search_console_2026_03.parquet (Search Console mid-panel month).

Time Window: March 1, 2026 to March 31, 2026 (2026-03).

In [13]:
import os
import duckdb
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")

# System environment variable me set kar do (sabse reliable tareeka)
os.environ["HF_TOKEN"] = hf_token

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

print("DuckDB connection established successfully.")

DuckDB connection established successfully.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Contract Question 2, 4 & 5: Field Buckets & ExclusionsFeatures: clicks, impressions, ctr, position, rolling_7d_avg_clicks (All historical metrics up to day $t-1$).Label / Proxy: is_high_performing_next_period (Binary label: whether clicks in the subsequent 7-day window exceed threshold).Context: url, query, date.Excluded (with reason): Same-day real-time conversion/revenue metrics and future click counts are deliberately excluded to avoid lookahead bias and feature leakage.

In [14]:
# Print field contract schema bucket summary
field_buckets = {
    "Features": ["clicks", "impressions", "ctr", "position", "rolling_7d_avg_clicks"],
    "Label": ["is_high_performing_next_period"],
    "Context": ["url", "query", "date"],
    "Excluded": ["future_clicks_7d", "conversion_revenue_realtime"]
}

for bucket, fields in field_buckets.items():
    print(f"[{bucket}]: {', '.join(fields)}")


[Features]: clicks, impressions, ctr, position, rolling_7d_avg_clicks
[Label]: is_high_performing_next_period
[Context]: url, query, date
[Excluded]: future_clicks_7d, conversion_revenue_realtime


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Verification Queries & Feature Leakage Trap Experiment:

Query 1: Verify Grain Uniqueness.

Query 2: Row Count and Date Span.

Query 3: Availability Check (IS TRUE).

Feature Frame + "Available When?" Annotations.

Deliberate Leakage Trap Experiment (Leakage Score vs Honest Score).

In [15]:
import os
import duckdb
import pandas as pd
from google.colab import userdata
from huggingface_hub import HfApi, hf_hub_download
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 1. Get Secret Token
hf_token = userdata.get("HF_TOKEN")

# 2. Find exact parquet filename in repository
api = HfApi()
repo_files = api.list_repo_files(repo_id="FlyRank/internship-warehouse", repo_type="dataset", token=hf_token)

target_file = None
for f in repo_files:
    if "2026-03" in f or "2026_03" in f:
        target_file = f
        break

if not target_file:
    parquet_files = [f for f in repo_files if f.endswith('.parquet')]
    target_file = parquet_files[0]

# 3. Download the exact file path
local_parquet_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename=target_file,
    repo_type="dataset",
    token=hf_token
)

con = duckdb.connect()
parquet_path = f"'{local_parquet_path}'"

# 4. Inspect Table Schema & Detect Numeric vs Text Columns
schema_df = con.execute(f"DESCRIBE SELECT * FROM {parquet_path}").df()

all_cols = schema_df['column_name'].tolist()

# Find text/entity columns and numeric metric columns safely
text_cols = [c for c in all_cols if c in ['url', 'page', 'content_hash_id', 'client_hash_id', 'query']]
date_cols = [c for c in all_cols if 'date' in c]
metric_cols = [c for c in all_cols if c in ['clicks', 'impressions', 'position', 'ctr', 'sessions', 'views', 'users', 'scroll_events']]

url_col = text_cols[0] if text_cols else all_cols[0]
date_col = date_cols[0] if date_cols else all_cols[1]

# Fallbacks for metrics
clicks_col = metric_cols[0] if metric_cols else all_cols[-2]
impressions_col = metric_cols[1] if len(metric_cols) > 1 else all_cols[-1]

print(f"Detected Schema -> Entity: '{url_col}', Date: '{date_col}', Metric1: '{clicks_col}', Metric2: '{impressions_col}'")

# -------------------------------------------------------------
# Query 1: Grain Verification (Duplicate Check)
# -------------------------------------------------------------
query_grain = f"""
SELECT {url_col}, {date_col}, COUNT(*) as cnt
FROM {parquet_path}
GROUP BY {url_col}, {date_col}
HAVING COUNT(*) > 1;
"""
df_grain = con.execute(query_grain).df()
print("\n1. Grain Duplicate Rows (Must be 0):", len(df_grain))

# -------------------------------------------------------------
# Query 2: Dataset Row Count and Date Span
# -------------------------------------------------------------
query_span = f"""
SELECT
    COUNT(*) as total_rows,
    MIN({date_col}) as start_date,
    MAX({date_col}) as end_date
FROM {parquet_path};
"""
df_span = con.execute(query_span).df()
print("\n2. Dataset Row Count & Date Span:")
print(df_span)

# -------------------------------------------------------------
# Query 3: Availability Verification (IS TRUE)
# -------------------------------------------------------------
has_avail = 'is_available' in all_cols
avail_clause = "WHERE is_available IS TRUE" if has_avail else ""
query_avail = f"""
SELECT COUNT(*) as surviving_rows
FROM {parquet_path}
{avail_clause};
"""
df_avail = con.execute(query_avail).df()
print("\n3. Available Rows (IS TRUE Check):")
print(df_avail)

# -------------------------------------------------------------
# Feature Frame & Leakage Trap Experiment (With Safe Numeric Casting)
# -------------------------------------------------------------
query_features = f"""
SELECT
    {url_col} as entity,
    {date_col} as date,
    TRY_CAST({clicks_col} AS DOUBLE) as feature_past_clicks,
    TRY_CAST({impressions_col} AS DOUBLE) as feature_past_impressions,
    AVG(TRY_CAST({clicks_col} AS DOUBLE)) OVER (
        PARTITION BY {url_col}
        ORDER BY {date_col}
        ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
    ) as feature_7d_avg_clicks
FROM {parquet_path}
LIMIT 5000;
"""
df_features = con.execute(query_features).df().fillna(0)

# Target Column
y_target = (df_features['feature_past_clicks'] > df_features['feature_past_clicks'].median()).astype(int)

# Leaked Column
df_features['leaked_label_column'] = y_target * 100 + 5

# Model WITH Leakage
X_leaked = df_features[['feature_past_clicks', 'feature_past_impressions', 'leaked_label_column']]
clf = RandomForestClassifier(random_state=42)
clf.fit(X_leaked, y_target)
leak_score = accuracy_score(y_target, clf.predict(X_leaked))
print(f"\n[LEAKAGE EXPERIMENT] Accuracy WITH Leaked Feature: {leak_score:.4f}")

# Model WITHOUT Leakage
X_honest = df_features[['feature_past_clicks', 'feature_past_impressions', 'feature_7d_avg_clicks']]
clf.fit(X_honest, y_target)
honest_score = accuracy_score(y_target, clf.predict(X_honest))
print(f"[HONEST EXPERIMENT] Accuracy WITHOUT Leaked Feature: {honest_score:.4f}")

Detected Schema -> Entity: 'client_hash_id', Date: 'report_date', Metric1: 'scroll_events', Metric2: 'month'

1. Grain Duplicate Rows (Must be 0): 1644

2. Dataset Row Count & Date Span:
   total_rows start_date   end_date
0     9841378 2026-03-01 2026-03-31

3. Available Rows (IS TRUE Check):
   surviving_rows
0         9841378


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


[LEAKAGE EXPERIMENT] Accuracy WITH Leaked Feature: 1.0000
[HONEST EXPERIMENT] Accuracy WITHOUT Leaked Feature: 1.0000


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Named Limitation of this Slice:

Single-Month Snapshot Limitation: This dataset slice is restricted to a single mid-panel month (2026-03). It does not account for long-term seasonality, holiday search trends, or major algorithm updates occurring in future quarters.

In [16]:
# Log limitation statement confirmation
limitation = "Dataset slice is constrained to 2026-03 and lacks long-term seasonal variance."
print("Named Limitation Recorded:", limitation)

Named Limitation Recorded: Dataset slice is constrained to 2026-03 and lacks long-term seasonal variance.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.